## Multiclass Text Classification

Sample dataset: https://www.kaggle.com/datasets/kaggle/san-francisco-crime-classification

## Create Session

In [1]:
from pyspark.sql import SparkSession

session = SparkSession.builder.appName("PysparkFTDS")\
    .config("spark.sql.shuffle.partitions", "50")\
    .config("spark.driver.maxResultSize", "5g")\
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .getOrCreate()

25/01/12 21:32:39 WARN Utils: Your hostname, Aeroflux-10.local resolves to a loopback address: 127.0.0.1; using 192.168.100.43 instead (on interface en0)
25/01/12 21:32:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/12 21:32:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read Data

In [2]:
# read csv
data = session.read.csv('train.csv', header=True, inferSchema=True)

# filter only selected columns
drop_list = ['Dates', 'DayOfWeek', 'PdDistrict', 'Resolution', 'Address', 'X', 'Y']
data = data.select([column for column in data.columns if column not in drop_list])
data.show(5)

+--------------+--------------------+
|      Category|            Descript|
+--------------+--------------------+
|      WARRANTS|      WARRANT ARREST|
|OTHER OFFENSES|TRAFFIC VIOLATION...|
|OTHER OFFENSES|TRAFFIC VIOLATION...|
| LARCENY/THEFT|GRAND THEFT FROM ...|
| LARCENY/THEFT|GRAND THEFT FROM ...|
+--------------+--------------------+
only showing top 5 rows



In [3]:
# show top 20 crime category
from pyspark.sql.functions import col
data.groupBy("Category") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------------------+------+
|            Category| count|
+--------------------+------+
|       LARCENY/THEFT|174900|
|      OTHER OFFENSES|126182|
|        NON-CRIMINAL| 92304|
|             ASSAULT| 76876|
|       DRUG/NARCOTIC| 53971|
|       VEHICLE THEFT| 53781|
|           VANDALISM| 44725|
|            WARRANTS| 42214|
|            BURGLARY| 36755|
|      SUSPICIOUS OCC| 31414|
|      MISSING PERSON| 25989|
|             ROBBERY| 23000|
|               FRAUD| 16679|
|FORGERY/COUNTERFE...| 10609|
|     SECONDARY CODES|  9985|
|         WEAPON LAWS|  8555|
|        PROSTITUTION|  7484|
|            TRESPASS|  7326|
|     STOLEN PROPERTY|  4540|
|SEX OFFENSES FORC...|  4388|
+--------------------+------+
only showing top 20 rows



In [4]:
# show top 20 description
data.groupBy("Descript") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+--------------------+-----+
|            Descript|count|
+--------------------+-----+
|GRAND THEFT FROM ...|60022|
|       LOST PROPERTY|31729|
|             BATTERY|27441|
|   STOLEN AUTOMOBILE|26897|
|DRIVERS LICENSE, ...|26839|
|      WARRANT ARREST|23754|
|SUSPICIOUS OCCURR...|21891|
|AIDED CASE, MENTA...|21497|
|PETTY THEFT FROM ...|19771|
|MALICIOUS MISCHIE...|17789|
|   TRAFFIC VIOLATION|16471|
|PETTY THEFT OF PR...|16196|
|MALICIOUS MISCHIE...|15957|
|THREATS AGAINST LIFE|14716|
|      FOUND PROPERTY|12146|
|ENROUTE TO OUTSID...|11470|
|GRAND THEFT OF PR...|11010|
|POSSESSION OF NAR...|10050|
|PETTY THEFT FROM ...|10029|
|PETTY THEFT SHOPL...| 9571|
+--------------------+-----+
only showing top 20 rows



## Pipeline Building

Here we are going to build a Pipeline that consist of

- RegexTokenizer
  - Reference:
    - https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.RegexTokenizer.html
    - https://george-jen.gitbook.io/data-science-and-apache-spark/tokenizer
    - https://medium.com/@harinata.0624/tokenizer-regextokenizer-in-pyspark-51a7c9b33132
- StopWordsRemover
  - Reference:
    - https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StopWordsRemover.html
    - https://george-jen.gitbook.io/data-science-and-apache-spark/stopwordremover
- CountVectorizer
  - Reference:
    - https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.CountVectorizer.html
    - https://george-jen.gitbook.io/data-science-and-apache-spark/countvectorizer
- StringIndexer
  - Reference:
    - https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html
    - https://george-jen.gitbook.io/data-science-and-apache-spark/stringindexer

**Note**: you can read each references to get better understanding.


In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, StringIndexer

# regular expression tokenizer
regexTokenizer = RegexTokenizer(inputCol="Descript", outputCol="words", pattern="\\W")

# stop words
add_stopwords = ["http","https","amp","rt","t","c","the"] 
stopwordsRemover = StopWordsRemover(inputCol="words", outputCol="filtered").setStopWords(add_stopwords)

# bag of words count
countVectors = CountVectorizer(inputCol="filtered", outputCol="features", vocabSize=10000, minDF=5)

# string indexer
label_stringIdx = StringIndexer(inputCol = "Category", outputCol = "label")
pipeline = Pipeline(stages=[regexTokenizer, stopwordsRemover, countVectors, label_stringIdx])

# Fit the pipeline to training documents.
pipelineFit = pipeline.fit(data)
dataset = pipelineFit.transform(data)
dataset.show(5)

25/01/12 21:32:46 WARN StopWordsRemover: Default locale set was [en_ID]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.


+--------------+--------------------+--------------------+--------------------+--------------------+-----+
|      Category|            Descript|               words|            filtered|            features|label|
+--------------+--------------------+--------------------+--------------------+--------------------+-----+
|      WARRANTS|      WARRANT ARREST|   [warrant, arrest]|   [warrant, arrest]|(809,[17,32],[1.0...|  7.0|
|OTHER OFFENSES|TRAFFIC VIOLATION...|[traffic, violati...|[traffic, violati...|(809,[11,17,35],[...|  1.0|
|OTHER OFFENSES|TRAFFIC VIOLATION...|[traffic, violati...|[traffic, violati...|(809,[11,17,35],[...|  1.0|
| LARCENY/THEFT|GRAND THEFT FROM ...|[grand, theft, fr...|[grand, theft, fr...|(809,[0,2,3,4,6],...|  0.0|
| LARCENY/THEFT|GRAND THEFT FROM ...|[grand, theft, fr...|[grand, theft, fr...|(809,[0,2,3,4,6],...|  0.0|
+--------------+--------------------+--------------------+--------------------+--------------------+-----+
only showing top 5 rows



## Split Data

In [6]:
(trainingData, testData) = dataset.randomSplit([0.7, 0.3], seed = 100)

print("Training Dataset Count: " + str(trainingData.count()))
print("Test Dataset Count: " + str(testData.count()))

Training Dataset Count: 614691


Test Dataset Count: 263358


### Model Training and Evaluation

References:
- https://spark.apache.org/docs/latest/ml-guide.html

#### Logistic Regression using Count Vector Features

In [8]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# create model
lr = LogisticRegression(maxIter=20, regParam=0.3, elasticNetParam=0)
lrModel = lr.fit(trainingData)
predictions = lrModel.transform(testData)

# evaluate model
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction")
evaluator.evaluate(predictions)

25/01/12 21:32:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/01/12 21:32:59 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


0.9719389853915912

#### Logistic Regression using TF-IDF Features

In [9]:
from pyspark.ml.feature import HashingTF, IDF

# create TF-IDF
hashingTF = HashingTF(inputCol="filtered", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features", minDocFreq=5) #minDocFreq: remove sparse terms

# create model pipeline
pipeline = Pipeline(stages=[regexTokenizer, stopwordsRemover, hashingTF, idf, label_stringIdx])
pipelineFit = pipeline.fit(data)
dataset = pipelineFit.transform(data)
(trainingData, testData) = dataset.randomSplit([0.7, 0.3], seed = 100)

# create model
lr = LogisticRegression(maxIter=20, regParam=0.3, elasticNetParam=0)
lrModel = lr.fit(trainingData)
predictions = lrModel.transform(testData)

# evaluate model
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction")
evaluator.evaluate(predictions)

0.9721362944012052

#### Cross-Validation

In [10]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# create model pipeline
pipeline = Pipeline(stages=[regexTokenizer, stopwordsRemover, countVectors, label_stringIdx])
pipelineFit = pipeline.fit(data)
dataset = pipelineFit.transform(data)
(trainingData, testData) = dataset.randomSplit([0.7, 0.3], seed=100)

# create model
lr = LogisticRegression(maxIter=20, regParam=0.3, elasticNetParam=0)

# create ParamGrid for Cross Validation
paramGrid = (
    ParamGridBuilder()
        .addGrid(lr.regParam, [0.1, 0.3, 0.5])  # regularization parameter
        .addGrid(lr.elasticNetParam, [0.0, 0.1, 0.2]) # Elastic Net Parameter (Ridge = 0)
        .build()
)

# create 5-fold CrossValidator
cv = CrossValidator(estimator=lr,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=5)
cvModel = cv.fit(trainingData)
predictions = cvModel.transform(testData)

# evaluate model
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction")
evaluator.evaluate(predictions)

0.9918960719818196

#### Naive Bayes

In [11]:
from pyspark.ml.classification import NaiveBayes

# create model
nb = NaiveBayes(smoothing=1)
model = nb.fit(trainingData)
predictions = model.transform(testData)

# evaluate model
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction")
evaluator.evaluate(predictions)

0.9934900857765636

#### Random Forrest

In [12]:
from pyspark.ml.classification import RandomForestClassifier

# create model
rf = RandomForestClassifier(labelCol="label",
                            featuresCol="features",
                            numTrees=100,
                            maxDepth=4,
                            maxBins=32)
rfModel = rf.fit(trainingData)
predictions = rfModel.transform(testData)

# evaluate model
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction")
evaluator.evaluate(predictions)

25/01/12 21:38:44 WARN MemoryStore: Not enough space to cache rdd_6664_5 in memory! (computed 13.0 MiB so far)
25/01/12 21:38:44 WARN MemoryStore: Not enough space to cache rdd_6664_7 in memory! (computed 19.6 MiB so far)
25/01/12 21:38:44 WARN BlockManager: Persisting block rdd_6664_5 to disk instead.
25/01/12 21:38:44 WARN MemoryStore: Not enough space to cache rdd_6664_4 in memory! (computed 5.7 MiB so far)
25/01/12 21:38:44 WARN BlockManager: Persisting block rdd_6664_4 to disk instead.
25/01/12 21:38:44 WARN MemoryStore: Not enough space to cache rdd_6664_6 in memory! (computed 19.6 MiB so far)
25/01/12 21:38:44 WARN BlockManager: Persisting block rdd_6664_6 to disk instead.
25/01/12 21:38:44 WARN BlockManager: Persisting block rdd_6664_7 to disk instead.
25/01/12 21:38:44 WARN MemoryStore: Not enough space to cache rdd_6664_3 in memory! (computed 13.0 MiB so far)
25/01/12 21:38:44 WARN BlockManager: Persisting block rdd_6664_3 to disk instead.
25/01/12 21:38:44 WARN MemoryStore: 

0.7322456702151177

In [13]:
# stop session
session.stop()